# Model comparison — SIH PS 26066

All satellite-only models trained on Jan–Dec **2015–2024** (80/20 time split). Metric: masked validation RMSE on `thetao`.

| Rank | Model | Overall RMSE | Decision |
|-----:|-------|-------------:|----------|
| 1 | **ViT** (masked MSE + L2) | **0.671 °C** | **kept** |
| 2 | ViT + $L_{\mathrm{grad}}$ | 0.686 °C | dropped |
| 3 | Thermocline-first | 0.713 °C | dropped |
| 4 | Structured embedding | 0.728 °C | dropped |
| 5 | Depth-cond (MSE) | 0.732 °C | dropped |
| 6 | Depth-cond + $L_{\mathrm{grad}}$ | 0.736 °C | dropped |
| 7 | TC-band ViT ($D_{tc}$ then $T$ on $[D_{tc},200]$) | 1.759 °C (band only) | dropped |

Probabilistic ViT (CRPS) was trained but never produced a validation RMSE; it is dropped.

The kept training notebook is `train_vit.ipynb`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
OUT = ROOT / "outputs" / "compare"
WEB = ROOT / "web" / "assets" / "compare"
OUT.mkdir(parents=True, exist_ok=True)
WEB.mkdir(parents=True, exist_ok=True)

DEPTHS = np.array([0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000], dtype=np.float32)

MODELS = {
    "ViT": {
        "loss": "masked MSE + L2",
        "keep": True,
        "overall": 0.671,
        "mean": 0.627,
        "rmse": np.array([0.428, 0.430, 0.438, 0.503, 0.591, 0.771, 0.946, 1.045, 1.046, 0.958, 0.708, 0.502, 0.337, 0.335, 0.360]),
    },
    "ViT + L_grad": {
        "loss": r"$L_T + \lambda L_{grad} + L_2$",
        "keep": False,
        "overall": 0.686,
        "mean": 0.637,
        "rmse": np.array([0.407, 0.412, 0.424, 0.510, 0.616, 0.820, 1.009, 1.088, 1.055, 0.950, 0.704, 0.503, 0.345, 0.343, 0.375]),
    },
    "Thermocline-first": {
        "loss": r"$L_{tc} + L_T + \lambda L_{grad}$",
        "keep": False,
        "overall": 0.713,
        "mean": 0.661,
        "rmse": np.array([0.451, 0.448, 0.445, 0.512, 0.623, 0.850, 1.068, 1.147, 1.097, 0.986, 0.715, 0.500, 0.345, 0.345, 0.383]),
    },
    "Structured embedding": {
        "loss": r"$L_T + L_{ground} + L_{decor} + L_2$",
        "keep": False,
        "overall": 0.728,
        "mean": 0.672,
        "rmse": np.array([0.406, 0.414, 0.428, 0.526, 0.648, 0.875, 1.091, 1.167, 1.123, 1.019, 0.745, 0.529, 0.357, 0.359, 0.393]),
    },
    "Depth-cond (MSE)": {
        "loss": "masked MSE + L2",
        "keep": False,
        "overall": 0.732,
        "mean": 0.679,
        "rmse": np.array([0.431, 0.437, 0.445, 0.528, 0.644, 0.883, 1.095, 1.159, 1.123, 1.034, 0.750, 0.525, 0.360, 0.364, 0.399]),
    },
    "Depth-cond + L_grad": {
        "loss": r"$L_T + \lambda L_{grad} + L_2$",
        "keep": False,
        "overall": 0.736,
        "mean": 0.682,
        "rmse": np.array([0.425, 0.429, 0.441, 0.534, 0.665, 0.918, 1.105, 1.163, 1.121, 1.023, 0.746, 0.528, 0.365, 0.364, 0.402]),
    },
    "TC-band ViT": {
        "loss": r"$D_{tc}$ then $T$ on $[D_{tc},200]$",
        "keep": False,
        "overall": 1.759,
        "mean": 1.795,
        "rmse": np.array([np.nan, 1.760, 1.852, 1.827, 1.864, 1.860, 1.819, 1.720, 1.735, 1.846, 1.670, np.nan, np.nan, np.nan, np.nan]),
    },
}

print("models:", list(MODELS))


In [ ]:
rows = []
for name, m in MODELS.items():
    tc = np.isin(DEPTHS, [50, 75, 100, 125, 150, 200])
    rows.append({
        "model": name,
        "loss": m["loss"],
        "kept": "yes" if m["keep"] else "no",
        "overall RMSE": m["overall"],
        "mean RMSE": m["mean"],
        "RMSE @100 m": float(m["rmse"][list(DEPTHS).index(100)]),
        "RMSE 50–200 m": float(np.sqrt(np.nanmean(m["rmse"][tc] ** 2))),
        "RMSE @1000 m": float(m["rmse"][-1]) if np.isfinite(m["rmse"][-1]) else float("nan"),
    })

df = pd.DataFrame(rows).sort_values("overall RMSE")
display(df.style.format({
    "overall RMSE": "{:.3f}", "mean RMSE": "{:.3f}",
    "RMSE @100 m": "{:.3f}", "RMSE 50–200 m": "{:.3f}", "RMSE @1000 m": "{:.3f}",
}))
df.to_csv(OUT / "summary.csv", index=False)
print("saved", OUT / "summary.csv")


In [ ]:
order = list(df["model"])
colors = {
    "ViT": "#1b4f72",
    "ViT + L_grad": "#2874a6",
    "Thermocline-first": "#117a65",
    "Structured embedding": "#b9770e",
    "Depth-cond (MSE)": "#6c3483",
    "Depth-cond + L_grad": "#5dade2",
    "TC-band ViT": "#922b21",
}

fig, ax = plt.subplots(figsize=(8.8, 4.8))
for i, name in enumerate(order):
    m = MODELS[name]
    ax.barh(i, m["overall"], color=colors[name], edgecolor="none", alpha=1.0 if m["keep"] else 0.55)
    tag = "  kept" if m["keep"] else ""
    ax.text(m["overall"] + 0.006, i, f'{m["overall"]:.3f}{tag}', va="center", fontsize=10)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.invert_yaxis()
ax.set_xlabel("overall validation RMSE (°C)")
ax.set_title("All models — overall RMSE (lower is better)")
ax.set_xlim(0, 2.05)
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "01_overall_rmse.png", dpi=150)
fig.savefig(WEB / "01_overall_rmse.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6.4))
styles = {
    "ViT": ("-", "o", 2.4),
    "ViT + L_grad": ("--", "s", 1.4),
    "Thermocline-first": ("-", "D", 1.4),
    "Structured embedding": (":", "^", 1.4),
    "Depth-cond (MSE)": ("-", "v", 1.4),
    "Depth-cond + L_grad": ("--", "P", 1.4),
    "TC-band ViT": (":", "X", 1.6),
}
for name, m in MODELS.items():
    ls, mk, lw = styles[name]
    ax.plot(
        m["rmse"], DEPTHS, linestyle=ls, marker=mk, markersize=5, linewidth=lw,
        color=colors[name], label=f'{name} ({m["overall"]:.3f})',
        alpha=1.0 if m["keep"] else 0.75,
    )
ax.invert_yaxis()
ax.set_xlabel("RMSE (°C)")
ax.set_ylabel("depth (m)")
ax.set_title("RMSE by depth")
ax.legend(fontsize=7.5, loc="lower right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "02_rmse_by_depth_sat_only.png", dpi=150)
fig.savefig(WEB / "02_rmse_by_depth_sat_only.png", dpi=150)
fig.savefig(OUT / "03_rmse_by_depth_all.png", dpi=150)
fig.savefig(WEB / "03_rmse_by_depth_all.png", dpi=150)
plt.show()


## Verdict

1. **ViT (MSE)** wins on every headline number: overall **0.671 °C**, mean 0.627 °C, thermocline band 50–200 m.
2. Thermocline gradient loss did not beat plain MSE (ViT 0.671 vs ViT+$L_{grad}$ 0.686; same pattern for depth-cond).
3. Extra architecture (two-stage thermocline, structured embedding, depth-conditioned decoder) all landed **0.71–0.74 °C**.
4. **TC-band ViT** (predict $D_{tc}$, then $T$ only on $[D_{tc}, 200]$) is worse still: **1.759 °C** on that band (Stage A $D_{tc}$ RMSE 28.4 m).
5. **Kept:** `train_vit.ipynb`. All other training notebooks removed.


In [ ]:
payload = {"depths": DEPTHS}
for name, m in MODELS.items():
    key = "".join(ch if ch.isalnum() else "_" for ch in name)
    payload[f"rmse_{key}"] = m["rmse"].astype(np.float32)
    payload[f"overall_{key}"] = np.float32(m["overall"])
np.savez(OUT / "metrics.npz", **payload)
print("saved", OUT)
print("\n=== Leaderboard ===")
for _, row in df.iterrows():
    mark = "KEEP" if row["kept"] == "yes" else "drop"
    print(f"  {row['overall RMSE']:.3f}  {row['model']:<22}  {mark}")
